# Push fleet model — demo & smoke test

Runs Paco's delivery-driven (**"push"**) fleet engine as a selectable AeroMAPS
efficiency model (`models_efficiency_push`) over the TP/RJ/NB/WB passenger
segments (`markets_push.yaml`).

**The push engine is calibrated to the end-2024 fleet**, so a scenario using it
**must pivot on 2024** (`prospection_start_year = 2025`). This demo uses the
ad-hoc 2000–2024 historic vectors from
`tutorials/13_change_the_prospection_start_year` (`inputs_2025.json`), wired via
`data/config_push_2025.yaml`.

Scope today: drop-in efficiency bridge (energy-per-ASK + 100 % drop-in shares).
Fleet counts / deliveries outputs and the plot migration are later phases.

In [ ]:
import warnings
import matplotlib.pyplot as plt
from aeromaps import create_process

warnings.simplefilter("ignore")  # quiet the reference-year notices + push feasibility logs

# CWD is the notebook dir; the config's paths are relative to itself.
process = create_process(configuration_file="data/config_push_2025.yaml")
process.compute()

print("prospection_start_year :", process.parameters.prospection_start_year)
print("last_historical_year   :", process.parameters.last_historical_year, "(engine pivot = 2024)")

## Per-segment energy-per-ASK

History (≤ 2024) is spliced from the AeroMAPS `energy_consumption_init / ask_init`
× the segment's 2024 energy/RPK shares; the projection (≥ 2025) is produced
bottom-up by the push engine from the surviving + delivered fleet. The 2024→2025
seam is smooth because both sides are now anchored on 2024.

In [ ]:
segments = ["turboprop", "regional_jet", "narrow_body", "wide_body"]
df = process.data["vector_outputs"]

print(f"{'segment':14s}{'2024':>10s}{'2025':>10s}{'2050':>10s}   (MJ/ASK)")
for mid in segments:
    s = df[f"energy_per_ask_without_operations_{mid}_dropin_fuel"]
    print(f"{mid:14s}{s.loc[2024]:10.4f}{s.loc[2025]:10.4f}{s.loc[2050]:10.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for mid in segments:
    s = df[f"energy_per_ask_without_operations_{mid}_dropin_fuel"].loc[2000:2050]
    ax.plot(s.index, s.values, label=mid.replace("_", " ").title())
ax.axvline(2024, ls="--", lw=1, color="grey")
ax.text(2024.3, ax.get_ylim()[1] * 0.95, "pivot 2024", color="grey", fontsize=8)
ax.set_xlabel("year")
ax.set_ylabel("energy per ASK [MJ/ASK]")
ax.set_title("Push fleet model — drop-in energy intensity by segment")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Guard: the push model requires a 2024 pivot

Selecting `models_efficiency_push` on any other pivot (e.g. the default
`prospection_start_year = 2020`) fails fast at build time, rather than silently
mis-dating the 2024 fleet.

In [ ]:
import tempfile
import yaml
from pathlib import Path
import aeromaps

RES = Path(aeromaps.__file__).parent / "resources" / "data"
bad_cfg = {  # no data.inputs -> prospection_start_year defaults to 2020
    "models": {
        "markets": {"markets_data_file": str(RES / "default_markets" / "markets_push.yaml")},
        "climate": {"climate_model_data_file": "default"},
        "energy": {
            "energy_carriers_model_data_file": "default",
            "resources_model_data_file": "default",
            "processes_model_data_file": "default",
        },
        "standards": [
            "models_traffic",
            "models_efficiency_push",
            "models_energy_without_fuel_effect",
            "models_offset",
            "models_emissions",
            "models_sustainability",
            "models_energy_cost",
            "models_operation_cost_top_down",
        ],
    },
}
with tempfile.NamedTemporaryFile("w", suffix=".yaml", delete=False) as f:
    yaml.safe_dump(bad_cfg, f)
    tmp = f.name

try:
    create_process(configuration_file=tmp)
    print("No error (unexpected).")
except ValueError as e:
    print("Guard fired as expected:\n")
    print(e)